# MedScribe Agent - MedGemma 4B Real Inference (Kaggle T4)

This notebook tests real MedGemma 4B inference with 4-bit quantization on a Kaggle T4 GPU.

## Prerequisites
- **Kaggle GPU**: Settings → Accelerator → GPU T4 x2
- **HuggingFace Account**: Accept MedGemma license at https://huggingface.co/google/medgemma-4b-it
- **HF Token**: Add as Kaggle Secret named `HF_TOKEN` (Settings → Secrets)

## 1. Install Dependencies

In [ ]:
%%time
# Install core dependencies
!pip install -q transformers>=4.50.0 bitsandbytes>=0.45.0 accelerate>=0.34.0
!pip install -q scispacy pyyaml
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

# Install HealthChain from GitHub
!pip install -q healthchain

print("\nDependencies installed.")

## 2. Check GPU & Login to HuggingFace

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    raise RuntimeError("No GPU detected! Enable GPU in Kaggle: Settings → Accelerator → GPU T4 x2")

In [ ]:
from huggingface_hub import login
import os

# On Kaggle, use Secrets to set HF_TOKEN
# On Colab/local, set HF_TOKEN env var or run `huggingface-cli login`
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HF_TOKEN")
    print("Loaded HF token from Kaggle Secrets")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")
    if hf_token:
        print("Loaded HF token from environment")
    else:
        print("No HF token found. Run: huggingface-cli login")
        hf_token = None

if hf_token:
    login(token=hf_token)
    print("Logged in to HuggingFace")

## 3. Load MedGemma 4B (4-bit Quantized)

In [ ]:
%%time
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "google/medgemma-4b-it"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Loading {MODEL_ID} with 4-bit NF4 quantization...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
print(f"Model loaded on {model.device}")

# Check VRAM usage
vram_used = torch.cuda.memory_allocated() / 1024**3
vram_total = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f"VRAM usage: {vram_used:.1f} GB / {vram_total:.1f} GB ({vram_used/vram_total*100:.0f}%)")

## 4. Test Raw Inference

Basic sanity check: send a simple medical question and verify the model responds coherently.

In [ ]:
%%time

messages = [
    {"role": "user", "content": "What is the ICD-10 code for community-acquired pneumonia?"}
]

inputs = tokenizer.apply_chat_template(
    messages, return_tensors="pt", add_generation_prompt=True
).to(model.device)

outputs = model.generate(inputs, max_new_tokens=256, do_sample=True, temperature=0.3)
response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

print("Prompt: What is the ICD-10 code for community-acquired pneumonia?")
print(f"\nResponse:\n{response}")

## 5. Test Clinical Reasoning Prompt

This tests the actual reasoning prompt template used by MedScribeAgent, and validates that the model returns parseable structured JSON.

In [ ]:
import json
import re
import time

SAMPLE_NOTE = """Patient is a 65-year-old male admitted for community-acquired pneumonia.
Past medical history significant for type 2 diabetes mellitus, hypertension,
and hyperlipidemia. Currently on metformin 1000mg BID, lisinopril 20mg daily,
and atorvastatin 40mg at bedtime. Chest X-ray shows right lower lobe infiltrate.
Started on azithromycin 500mg IV and ceftriaxone 1g IV.
Labs show WBC 14.2, glucose 185 mg/dL, HbA1c 7.8%.
Assessment: Community-acquired pneumonia, uncontrolled type 2 diabetes."""

REASONING_PROMPT = """You are a clinical coding agent. Given the clinical note below, extract ALL diagnoses, procedures, and medications. For each, provide:
1. Entity text (exact quote from note)
2. Suggested ICD-10 or CPT code
3. Confidence (high/medium/low)
4. Any documentation gaps or concerns

If NLP-extracted entities are provided, cross-reference them with your own analysis and flag any discrepancies.

Respond ONLY in valid JSON with this schema:
{"diagnoses": [{"text": str, "icd10": str, "confidence": str, "gaps": str}], "procedures": [...], "medications": [...], "discrepancies": [...]}
"""

messages = [
    {"role": "system", "content": REASONING_PROMPT},
    {"role": "user", "content": f"Clinical Note:\n{SAMPLE_NOTE}"},
]

inputs = tokenizer.apply_chat_template(
    messages, return_tensors="pt", add_generation_prompt=True
).to(model.device)

start = time.time()
outputs = model.generate(inputs, max_new_tokens=2048, do_sample=True, temperature=0.3)
latency = time.time() - start

raw_response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

print(f"Inference latency: {latency:.1f}s")
print(f"Output tokens: {outputs.shape[-1] - inputs.shape[-1]}")
print(f"\nRaw response:\n{raw_response}")

In [ ]:
# Parse the JSON response (with fallback regex extraction)
def parse_json_response(response, fallback):
    """Extract JSON from model response, handling markdown code blocks."""
    try:
        return json.loads(response)
    except (json.JSONDecodeError, TypeError):
        pass
    json_match = re.search(r"\{[\s\S]*\}", response)
    if json_match:
        try:
            return json.loads(json_match.group())
        except json.JSONDecodeError:
            pass
    print(f"WARNING: Could not parse JSON from response")
    return fallback

parsed = parse_json_response(
    raw_response,
    {"diagnoses": [], "procedures": [], "medications": [], "discrepancies": []}
)

print("Parsed JSON:")
print(json.dumps(parsed, indent=2))

# Validate structure
print(f"\n--- Validation ---")
print(f"Has 'diagnoses': {'diagnoses' in parsed} ({len(parsed.get('diagnoses', []))} items)")
print(f"Has 'procedures': {'procedures' in parsed} ({len(parsed.get('procedures', []))} items)")
print(f"Has 'medications': {'medications' in parsed} ({len(parsed.get('medications', []))} items)")

for dx in parsed.get("diagnoses", []):
    has_text = "text" in dx
    has_icd10 = "icd10" in dx
    has_confidence = "confidence" in dx
    print(f"  Dx: {dx.get('text', 'N/A'):40s} ICD-10: {dx.get('icd10', 'N/A'):10s} Conf: {dx.get('confidence', 'N/A'):8s} {'✓' if all([has_text, has_icd10, has_confidence]) else '✗ MISSING FIELDS'}")

## 6. Test Resolution Prompt

Test the discrepancy resolution prompt with simulated NLP vs LLM discrepancies.

In [ ]:
RESOLUTION_PROMPT = """You are a clinical coding agent reviewing a discrepancy between NLP extraction and your prior analysis.

Given the original clinical note, the discrepancy details, and relevant patient history (FHIR resources), determine the correct coding.

For each resolved item, provide:
1. Final entity text
2. Final ICD-10 or CPT code
3. Resolution reasoning
4. Confidence (high/medium/low)

Respond ONLY in valid JSON with this schema:
{"resolved": [{"text": str, "code": str, "reasoning": str, "confidence": str}]}
"""

discrepancies = [
    {"type": "nlp_only", "entity": "right lower lobe infiltrate", "reason": "Found by NLP but not confirmed by LLM"},
    {"type": "llm_only", "entity": "uncontrolled diabetes", "icd10": "E11.65", "reason": "LLM inferred but NLP did not extract"}
]

user_content = (
    f"Original Clinical Note:\n{SAMPLE_NOTE}\n\n"
    f"Discrepancies to resolve:\n{json.dumps(discrepancies, indent=2)}\n\n"
    f"Relevant Patient History (FHIR):\n"
    f"Observation: HbA1c 7.8% (2 days ago)\n"
    f"Condition: Type 2 diabetes mellitus (active, diagnosed 2019)"
)

messages = [
    {"role": "system", "content": RESOLUTION_PROMPT},
    {"role": "user", "content": user_content},
]

inputs = tokenizer.apply_chat_template(
    messages, return_tensors="pt", add_generation_prompt=True
).to(model.device)

start = time.time()
outputs = model.generate(inputs, max_new_tokens=1024, do_sample=True, temperature=0.3)
latency = time.time() - start

raw = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print(f"Resolution latency: {latency:.1f}s")
print(f"\nRaw response:\n{raw}")

resolution = parse_json_response(raw, {"resolved": []})
print(f"\nParsed resolution:")
print(json.dumps(resolution, indent=2))

## 7. Test via MedGemmaClient Class

Now test through the actual `MedGemmaClient` class to verify the full code path works.

In [ ]:
import sys
import os

# Add project root to path (adjust for your setup)
# On Kaggle: clone the repo first, then add to path
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Force real mode since we have a GPU
os.environ["MEDGEMMA_MODE"] = "real"
os.environ["MEDGEMMA_MODEL"] = "google/medgemma-4b-it"

import logging
logging.basicConfig(level=logging.INFO)

from medscribe.src.models.medgemma import create_client

client = create_client()
print(f"Client type: {type(client).__name__}")

In [ ]:
%%time

# Test reason_over_note through the client
result = client.reason_over_note(SAMPLE_NOTE)
print("reason_over_note result:")
print(json.dumps(result, indent=2))

In [ ]:
%%time

# Test reason_with_resolution through the client
discrepancies = [
    {"type": "nlp_only", "entity": "right lower lobe infiltrate", "reason": "Found by NLP but not confirmed by LLM"}
]
resolution = client.reason_with_resolution(SAMPLE_NOTE, discrepancies, patient_history="Condition: Type 2 DM (active)")
print("reason_with_resolution result:")
print(json.dumps(resolution, indent=2))

## 8. End-to-End Pipeline with Real MedGemma

Run the full MedScribeAgent pipeline: NLP extraction → real MedGemma reasoning → validation → output.

In [ ]:
%%time
from healthchain.io import Document
from medscribe.src.pipeline.coding_pipeline import build_coding_pipeline_simple
from medscribe.src.agent.orchestrator import MedScribeAgent

# Build components
pipeline = build_coding_pipeline_simple()

agent = MedScribeAgent(
    coding_pipeline=pipeline,
    medgemma_client=client,
    fhir_gateway=None,
)

# Run full agent pipeline
output = agent.run(SAMPLE_NOTE, patient_id="Patient/demo-001")

print(f"\n{'='*60}")
print("AGENT RUN COMPLETE (REAL MEDGEMMA)")
print(f"{'='*60}")

In [ ]:
# Inspect results
print(f"NLP Entities: {len(output['nlp_entities'])}")
for e in output['nlp_entities']:
    print(f"  {e.get('text', 'N/A'):30s} [{e.get('label', 'N/A')}]")

print(f"\nLLM Diagnoses: {len(output['llm_reasoning'].get('diagnoses', []))}")
for dx in output['llm_reasoning'].get('diagnoses', []):
    print(f"  {dx.get('text', 'N/A'):40s} ICD-10: {dx.get('icd10', 'N/A'):10s} [{dx.get('confidence', 'N/A')}]")

print(f"\nDiscrepancies: {len(output['discrepancies'])}")
for d in output['discrepancies']:
    print(f"  [{d['type']}] {d.get('entity', 'N/A')}: {d.get('reason', '')}")

print(f"\nCDS Cards: {len(output['cds_cards'])}")
for card in output['cds_cards']:
    print(f"  [{card.indicator.value:8s}] {card.summary}")

## 9. Latency Benchmark

Run multiple notes and measure average latency.

In [ ]:
TEST_NOTES = [
    SAMPLE_NOTE,
    """Patient is a 45-year-old female presenting with chest pain and shortness of breath.
    EKG shows ST elevation in leads II, III, aVF. Troponin elevated at 2.5 ng/mL.
    History of hypertension, hyperlipidemia. Current medications: aspirin, metoprolol, atorvastatin.
    Assessment: Acute inferior STEMI. Plan: emergent cardiac catheterization.""",
    """72-year-old female with worsening dyspnea and bilateral lower extremity edema.
    Echocardiogram shows EF 25%, severe mitral regurgitation.
    BNP 1850 pg/mL. CXR with bilateral pleural effusions.
    History: CHF, atrial fibrillation, CKD stage 3. On furosemide, carvedilol, apixaban.
    Assessment: Acute decompensated heart failure. Started IV diuresis.""",
]

latencies = []
for i, note in enumerate(TEST_NOTES):
    start = time.time()
    result = client.reason_over_note(note)
    elapsed = time.time() - start
    latencies.append(elapsed)
    n_dx = len(result.get("diagnoses", []))
    print(f"Note {i+1}: {elapsed:.1f}s | {n_dx} diagnoses")

print(f"\nAverage latency: {sum(latencies)/len(latencies):.1f}s")
print(f"Min: {min(latencies):.1f}s | Max: {max(latencies):.1f}s")
print(f"Target: <15s per note on T4 GPU")

## 10. Summary

Record results for the project writeup.

In [ ]:
print("MedGemma 4B Inference Test Results")
print("=" * 50)
print(f"Model: {MODEL_ID}")
print(f"Quantization: 4-bit NF4 (bitsandbytes)")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"Avg inference latency: {sum(latencies)/len(latencies):.1f}s")
print(f"JSON parse success: check cells above")
print(f"End-to-end pipeline: {'PASS' if output.get('cds_cards') else 'FAIL'}")